In [1]:
import sklearn.datasets as datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

import mlgrad.regr as regr
import mlgrad.funcs as funcs
import mlgrad.funcs2 as funcs2
import mlgrad.models as models
import mlgrad.loss as loss


In [2]:
X_all, y_all = datasets.fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
print(X_all.shape, y_all.shape)


(70000, 784) (70000,)


In [3]:
X_all = X_all.astype("d")
X_all /= 255.0 
y_all = np.array([int(x) for x in y_all])
cls_ids = [int(x) for x in np.unique(y_all)]

Xt, yt = X_all[:50000], y_all[:50000]
Xc, yc = X_all[50000:], y_all[50000:]
Xt = np.ascontiguousarray(Xt)
Xc = np.ascontiguousarray(Xc)
yt = np.ascontiguousarray(yt)
yc = np.ascontiguousarray(yc)


In [ ]:
cls_dict = {}
for i in range(10):
    Xt_i = Xt[yt == i]
    yt_i = np.ones(len(Xt_i))
    Xc_i = Xc[yc == i]
    yc_i = np.ones(len(Xc_i))
    for j in range(10):
        if i == j:
            continue
        Xt_j = Xt[yt == j]
        yt_j = -np.ones(len(Xt_j))
        Xt_ij = np.vstack((Xt_i, Xt_j))
        Xt_ij = np.ascontiguousarray(Xt_ij)
        yt_ij = np.concatenate((yt_i, yt_j))
        yt_ij = np.ascontiguousarray(yt_ij)

        Xc_j = Xc[yc == j]
        yc_j = -np.ones(len(Xc_j))
        Xc_ij = np.vstack((Xc_i, Xc_j))
        Xc_ij = np.ascontiguousarray(Xc_ij)
        yc_ij = np.concatenate((yc_i, yc_j))
        yc_ij = np.ascontiguousarray(yc_ij)

        mod = models.LinearModel(Xt_ij.shape[0])
        # mod.use_regularizer(funcs2.SquareNorm(), 0.01)
        reg = regr.regression(Xt_ij, yt_ij, mod, 
                              loss.MarginLoss(funcs.Hinge(1.0)), 
                              h=0.01, n_iter=500, tol=1.0e-4)
        print(i, j, 
              accuracy_score(yt_ij, np.sign(mod.evaluate(Xt_ij))),
              accuracy_score(yc_ij, np.sign(mod.evaluate(Xc_ij))),
             )
        cls_dict[(i,j)] = mod

0 1 0.9989632422243166 0.9997601918465228
0 2 0.9992929292929293 0.998747808665164
0 3 0.9985049337187282 0.9985041136873598
0 4 0.9994893269328976 0.9989837398373984
0 5 0.9991523627887264 0.9973530968766543
0 6 0.9993928968936557 0.9974332648870636
0 7 0.9985158800831108 0.9995108828564441
0 8 0.9990791896869244 0.9977238239757208
0 9 0.9995967741935484 0.9969550875412332
1 0 0.9995287464655985 0.9971223021582734
1 2 0.9992485440541048 0.9983416252072969
1 3 0.9991650431394378 0.9985845718329794
1 4 0.999335674290595 0.9983189241114313
1 5 0.9998036135113905 0.9985022466300549
1 6 0.9989650954934612 0.9958777885548011
1 7 0.999262876623975 0.9988417882788974
1 8 0.9995247148288974 0.9983261597321855
1 9 0.9992499531220701 0.9971216118973375
2 0 0.9987878787878788 0.9957425494615577
2 1 0.9992485440541048 0.9971570717839374
2 3 0.9980137054325157 0.9977843426883308
2 4 0.9996947186323395 0.9982442939553549
2 5 0.9996833438885371 0.9958213632802299
2 6 0.9991934670833753 0.998479858120

In [6]:
def multi_classify(cls_dict, X):
    ys = []
    for i in range(10):
        y_i = np.zeros(len(X))
        for j in range(10):
            if i == j:
                continue
            cls = cls_dict[(i,j)]
            y = np.sign(cls.evaluate(X))
            y = (y + 1) / 2
            y_i += y
        ys.append(y_i)
    ys = tuple(ys)
    Ys = np.c_[ys]
    ret = np.argmax(Ys, axis=1)
    return ret

In [7]:
y = multi_classify(cls_dict, Xt)
print(accuracy_score(y, yt))
y = multi_classify(cls_dict, Xc)
print(accuracy_score(y, yc))

0.14634
0.15055


In [8]:
def transform(cls_dict, X):
    ys = []
    for i in range(10):
        for j in range(10):
            if i == j:
                continue
            cls = cls_dict[(i,j)]
            y = cls.predict(X)
            ys.append(y)
    ys = tuple(ys)
    U = np.c_[ys]
    return U

In [9]:
Ut = transform(cls_dict, Xt)
Uc = transform(cls_dict, Xc)

In [10]:
cls = LogisticRegression(max_iter=500, penalty="l2", C=1.0)
cls.fit(Ut, yt)

/usr/lib/python3/dist-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,500
,multi_class,'deprecated'


In [11]:
print(accuracy_score(yt, cls.predict(Ut)))
print(accuracy_score(yc, cls.predict(Uc)))

0.4376
0.42835
